In [3]:

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [4]:
sns.set_theme(style="whitegrid")
FILE = "/home/student/Downloads/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv" 

In [5]:
#0. Load & clean

df = pd.read_csv(FILE)
print("Shape:", df.shape)


Shape: (7043, 21)


In [6]:
df.drop("customerID", axis=1, inplace=True)

In [7]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)


In [8]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})


In [9]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [10]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = X.select_dtypes(
    include=["object"]
).columns


print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

Categorical columns:
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [11]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (5634, 19)
Testing data: (1409, 19)


In [21]:
model = RandomForestClassifier(
    n_estimators=20,
    random_state=42,
    class_weight="balanced"
)

In [22]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [23]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
y_pred = pipeline.predict(X_test)

In [25]:
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.7735982966643009

Confusion Matrix:
[[922 113]
 [206 168]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1035
           1       0.60      0.45      0.51       374

    accuracy                           0.77      1409
   macro avg       0.71      0.67      0.68      1409
weighted avg       0.76      0.77      0.76      1409



In [26]:
from sklearn.model_selection import GridSearchCV

In [27]:
# ==========================================
# HYPERPARAMETER TUNING - RANDOM FOREST
# ==========================================

# 1. Define hyperparameters to test
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5, 10]
}

# 2. Create GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

# 3. Train using hyperparameter tuning
print("\nStarting Hyperparameter Tuning...")

grid_search.fit(X_train, y_train)

# 4. Display the best hyperparameters
print("\nBest Hyperparameters:")
print(grid_search.best_params_)

# 5. Display the best cross-validation F1-score
print("\nBest Cross-Validation F1-Score:")
print(grid_search.best_score_)

# 6. Get the best tuned model
tuned_model = grid_search.best_estimator_

# 7. Predict using the tuned model
y_pred_tuned = tuned_model.predict(X_test)

# 8. Evaluate the tuned model
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("\n========== TUNED RANDOM FOREST RESULTS ==========")

print("\nTuned Model Accuracy:")
print(tuned_accuracy)

print("\nTuned Model Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nTuned Model Classification Report:")
print(classification_report(y_test, y_pred_tuned))


Starting Hyperparameter Tuning...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best Hyperparameters:
{'model__max_depth': 10, 'model__min_samples_split': 10, 'model__n_estimators': 50}

Best Cross-Validation F1-Score:
0.6352666707645593

========== TUNED RANDOM FOREST RESULTS ==========

Tuned Model Accuracy:
0.7629524485450674

Tuned Model Confusion Matrix:
[[801 234]
 [100 274]]

Tuned Model Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.73      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [20]:
# ==========================================
# COMPARISON: NORMAL VS TUNED MODEL
# ==========================================

print("\n========== MODEL COMPARISON ==========")

print("\nNormal Random Forest Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nTuned Random Forest Accuracy:")
print(accuracy_score(y_test, y_pred_tuned))

print("\nNormal Random Forest Classification Report:")
print(classification_report(y_test, y_pred))

print("\nTuned Random Forest Classification Report:")
print(classification_report(y_test, y_pred_tuned))


========== MODEL COMPARISON ==========

Normal Random Forest Accuracy:
0.7828246983676366

Tuned Random Forest Accuracy:
0.7629524485450674

Normal Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1035
           1       0.62      0.47      0.53       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409


Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.73      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

In [28]:
model_1 = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

In [29]:
# Decision Tree Pipeline
pipeline_1 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model_1)
])


# Train Decision Tree
pipeline_1.fit(X_train, y_train)


# Predict using Decision Tree
y_pred_1 = pipeline_1.predict(X_test)


# Decision Tree Accuracy
accuracy_1 = accuracy_score(y_test, y_pred_1)

print("\nDecision Tree Accuracy:", accuracy_1)


# Decision Tree Confusion Matrix
print("\nDecision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_1))


# Decision Tree Classification Report
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, y_pred_1))


Decision Tree Accuracy: 0.7281760113555713

Decision Tree Confusion Matrix:
[[845 190]
 [193 181]]

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.82      0.82      1035
           1       0.49      0.48      0.49       374

    accuracy                           0.73      1409
   macro avg       0.65      0.65      0.65      1409
weighted avg       0.73      0.73      0.73      1409

